# AI-vs-real detector — Phase 5 Step 3: stronger backbone on GPU

Trains **ConvNeXt-Tiny** and **ResNet50** (vs the M1 baseline's ResNet18) on the same
leakage-safe split, with the same augmentation, and evaluates each on the **held-out
Midjourney** generator. Goal: beat the ResNet18 numbers on mj6 (baseline 57.5% → augmented 62.6% AI recall).

### The only manual step
1. In Colab: **Runtime → Change runtime type → GPU (T4)**.
2. Upload `data_bundle.tar` (~1.4 GB) to your Google Drive at **`MyDrive/ai_detector/data_bundle.tar`**.
   (Any path is fine — just update `DRIVE_TARBALL` in the config cell.)
3. **Runtime → Run all.** Approve the Drive-mount popup when it appears. Then wait.

Everything else — code, data, training, evaluation, saving results — is automated below.

## 1. Confirm we have a GPU

In [ ]:
import torch
print('torch', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU! Runtime -> Change runtime type -> GPU, then Run all again.'
print('GPU:', torch.cuda.get_device_name(0))
!nvidia-smi -L

## 2. Clone the project code (from GitHub)

In [ ]:
import os
REPO = 'https://github.com/samvithvaddiparthi-ship-it/Ai-Real-Image-Detector.git'
os.chdir('/content')
if not os.path.isdir('Ai-Real-Image-Detector'):
    !git clone $REPO
os.chdir('/content/Ai-Real-Image-Detector')
!git pull -q
os.environ['PYTHONPATH'] = os.getcwd()  # so `from src....` imports resolve
print('cwd:', os.getcwd())

## 3. Mount Drive and unpack the data
Approve the popup. This recreates `data/raw/` (the 18.5k images) + `data/splits.csv` inside the repo.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_TARBALL = '/content/drive/MyDrive/ai_detector/data_bundle.tar'  # <- edit if you put it elsewhere
assert os.path.exists(DRIVE_TARBALL), (
    f'Not found: {DRIVE_TARBALL}\n'
    'Upload data_bundle.tar to that Drive path, or fix DRIVE_TARBALL above.')

print('extracting (~1.4 GB, a minute or two)...')
!tar -xf "$DRIVE_TARBALL" -C /content/Ai-Real-Image-Detector

import pandas as pd
df = pd.read_csv('data/splits.csv')
print('\nsplit sizes:')
print(df.groupby(['split', 'label_a']).size())
assert df.split.nunique() == 4, 'splits.csv looks wrong'
print('\ndata OK:', len(df), 'rows')

## 4. Config
GPU trains fast, so we do both backbones and compare. Free T4 fits batch 96 at 224px.
To try just one, trim `ARCHS_TO_TRAIN`.

In [ ]:
ARCHS_TO_TRAIN = ['convnext_tiny', 'resnet50']
TAG = 'colab'
BATCH = 96
EPOCHS = 15
WORKERS = 2   # Colab has ~2 vCPUs; the JPEG augmentation is CPU-side
LR = 1e-4
print('will train:', ARCHS_TO_TRAIN)

## 5. Train + evaluate each backbone
Each: ~15 epochs with val-based early stopping (same loop as local), then the full
evaluation including held-out Midjourney. Expect roughly 20–40 min per backbone.

In [ ]:
for ARCH in ARCHS_TO_TRAIN:
    print('\n' + '=' * 72 + f'\nTRAIN  {ARCH}\n' + '=' * 72)
    !python src/train.py --arch {ARCH} --tag {TAG} --batch-size {BATCH} --epochs {EPOCHS} --num-workers {WORKERS} --lr {LR}
    print('\n' + '=' * 72 + f'\nEVAL   {ARCH}\n' + '=' * 72)
    !python src/evaluate.py --ckpt models/{ARCH}_{TAG}.pth

## 6. Comparison table (the number that matters: held-out mj6 AI recall)

In [ ]:
import json, glob
REFERENCE = {
    'resnet18_baseline (M1)':  {'test_acc': 0.954, 'mj6_recall': 0.575, 'mj6_prec': 0.928},
    'resnet18_augmented (M1)': {'test_acc': 0.930, 'mj6_recall': 0.626, 'mj6_prec': 0.918},
}
rows = dict(REFERENCE)
for ARCH in ARCHS_TO_TRAIN:
    p = f'reports/eval_metrics_{ARCH}_{TAG}.json'
    if os.path.exists(p):
        d = json.load(open(p))
        rows[f'{ARCH}_{TAG} (GPU)'] = {
            'test_acc': d['test']['accuracy'],
            'mj6_recall': d['gen_holdout']['recall_ai'],
            'mj6_prec': d['gen_holdout']['precision_ai'],
        }
print(f"{'model':<28} {'test_acc':>9} {'mj6_recall':>11} {'mj6_prec':>9}")
print('-' * 60)
for name, m in rows.items():
    print(f"{name:<28} {m['test_acc']:>9.3f} {m['mj6_recall']:>11.3f} {m['mj6_prec']:>9.3f}")

## 7. Save models + metrics back to Drive (Colab is ephemeral)

In [ ]:
OUT = '/content/drive/MyDrive/ai_detector/results'
os.makedirs(OUT, exist_ok=True)
for ARCH in ARCHS_TO_TRAIN:
    !cp -v models/{ARCH}_{TAG}.pth {OUT}/ 2>/dev/null
    !cp -v reports/eval_metrics_{ARCH}_{TAG}.json {OUT}/ 2>/dev/null
    !cp -v reports/confusion_{ARCH}_{TAG}_*.png {OUT}/ 2>/dev/null
    !cp -v reports/train_history_{ARCH}_{TAG}.csv {OUT}/ 2>/dev/null
print('\nsaved to', OUT)
!ls -lh {OUT}

## Done — what to do next
- Read the **comparison table** in step 6. If a GPU backbone beats `resnet18_augmented`'s
  **62.6%** mj6 recall, that's our win.
- The trained `.pth` files are in your Drive under `ai_detector/results/`.
- Paste the step-6 table (and the two EVAL blocks from step 5) back to Claude, and we'll
  pick the winner, optionally calibrate the decision threshold, and move toward Phase 6
  (deployment).